# Exploring FHIR R4 Bundles with pandas

This notebook loads FHIR Bundles from S3 (or a local folder), turns them into one pandas DataFrame per resource type, and shows example queries.

**Using a notebook:** run cells one at a time with **Shift+Enter**, or use *Run → Run All Cells*. Cells share state, so run them in order from the top. If things get confusing, use *Kernel → Restart Kernel and Run All Cells*.

It runs on the synthetic sample data in `data/sample/` by default. To use your own data, change `SOURCE` in the **Parameters** cell below.

In [6]:
# Reload cup_fhir automatically when its source files change (no kernel restart needed)
%load_ext autoreload
%autoreload 2

import pandas as pd

from cup_fhir import (
    LOINC, SNOMED,
    load_bundles, validate_bundles, to_dataframes, populated,
    codes, has_code, has_category, identifier_value,
    patient_names_and_ages, with_patient, latest_per_patient,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Parameters

In [7]:
# Where to load from: an s3:// URI or a local folder (relative to this notebook)
SOURCE = "../data/sample"
# SOURCE = "s3://my-bucket/path/to/fhir/"

## 1. Load the bundles

`load_bundles` scans `SOURCE` for `.json`, `.ndjson` and gzipped files and keeps everything whose `resourceType` is `Bundle`. Problem files are reported rather than stopping the load.

In [8]:
result = load_bundles(SOURCE)
result

LoadResult(2 bundles, 1 files without bundles, 0 errors)

In [9]:
# Files that couldn't be read, and files that contained no Bundles
display(result.errors)
display(result.files_without_bundles)

[]

['../data/sample/not-a-bundle.json']

### Optional: validate against the FHIR R4 models

This parses each Bundle into a typed `fhir.resources` object. Anything listed in `invalid` is still usable as raw JSON through `result.bundles`.

In [10]:
valid, invalid = validate_bundles(result.bundles)
print(f"{len(valid)} valid, {len(invalid)} invalid")
for key, err in invalid.items():
    print(key, "\n", err, "\n")

2 valid, 0 invalid


In [11]:
# Typed objects give attribute access and tab-completion
bundle = next(iter(valid.values()))
bundle.type, len(bundle.entry), [e.resource.get_resource_type() for e in bundle.entry][:5]

('collection', 3, ['Patient', 'Condition', 'Observation'])

## 2. One DataFrame per resource type

Nested objects become dotted columns (`subject.reference`, `valueQuantity.value`). Arrays such as `code.coding`, `name` and `identifier` stay as list columns, and the `cup_fhir` helpers query those.

Each row also gets `_source` (which Bundle it came from) and `_fullUrl` (needed to resolve references).

In [12]:
dfs = to_dataframes(result.bundles)
{resource_type: len(df) for resource_type, df in dfs.items()}

{'Condition': 3, 'MedicationRequest': 1, 'Observation': 5, 'Patient': 3}

In [13]:
patients     = patient_names_and_ages(dfs["Patient"])   # adds family, given, age
observations = dfs.get("Observation", pd.DataFrame())
conditions   = dfs.get("Condition", pd.DataFrame())
medreqs      = dfs.get("MedicationRequest", pd.DataFrame())

In [14]:
# Which fields does this dataset actually fill in? (fraction of rows with a value)
populated(observations)

_source                 1.0
_fullUrl                1.0
resourceType            1.0
id                      1.0
status                  1.0
category                1.0
effectiveDateTime       1.0
code.coding             1.0
code.text               1.0
subject.reference       1.0
valueQuantity.value     1.0
valueQuantity.unit      1.0
valueQuantity.system    1.0
valueQuantity.code      1.0
dtype: float64

## 3. Patients

In [15]:
patients[["id", "family", "given", "gender", "birthDate", "age"]]

,id,family,given,gender,birthDate,age
0,carol,Brown,Carol,female,1948-11-30,77
1,alice,Nguyen,Alice May,female,1955-06-14,71
2,bob,Smith,Bob,male,1980-01-02,46


In [16]:
# Women aged 65 and over
patients.query("gender == 'female' and age >= 65")[["id", "family", "given", "age"]]

,id,family,given,age
0,carol,Brown,Carol,77
1,alice,Nguyen,Alice May,71


In [17]:
# Born in a date range
patients[patients["birthDate"].between("1950-01-01", "1959-12-31")][["id", "family", "birthDate"]]

,id,family,birthDate
1,alice,Nguyen,1955-06-14


In [18]:
# Pull out an identifier by its system URI (change this to your MRN / Medicare / IHI system)
MRN_SYSTEM = "http://example.org/fhir/mrn"
patients["mrn"] = identifier_value(patients["identifier"], MRN_SYSTEM)
patients[patients["mrn"] == "MRN001"][["id", "mrn", "family"]]

,id,mrn,family
1,alice,MRN001,Nguyen


## 4. Observations

Coded fields are `CodeableConcept`s: a list of codings, each with a `system` (e.g. LOINC) and a `code`. `has_code` gives a True/False mask to filter on.

In [19]:
# Parse dates once. utc=True copes with mixed timezone offsets.
observations["effective"] = pd.to_datetime(observations["effectiveDateTime"], format="ISO8601", utc=True)

# What kinds of observation are there?
observations["code.text"].value_counts()

code.text
Hemoglobin A1c/Hemoglobin.total in Blood    4
Body weight                                 1
Name: count, dtype: int64

In [20]:
# All HbA1c results (LOINC 4548-4)
hba1c = observations[has_code(observations["code.coding"], "4548-4", LOINC)]
hba1c[["subject.reference", "effective", "valueQuantity.value", "valueQuantity.unit"]]

,subject.reference,effective,valueQuantity.value,valueQuantity.unit
0,Patient/carol,2024-11-20 00:00:00+00:00,9.1,%
1,urn:uuid:8f0c6a3e-0d1b-4f3a-9d61-1a2b3c4d5e01,2024-03-04 22:30:00+00:00,7.8,%
2,urn:uuid:8f0c6a3e-0d1b-4f3a-9d61-1a2b3c4d5e01,2025-02-10 23:00:00+00:00,8.4,%
3,urn:uuid:8f0c6a3e-0d1b-4f3a-9d61-1a2b3c4d5e02,2025-08-31 22:15:00+00:00,5.6,%


In [21]:
# HbA1c above 8%
hba1c[hba1c["valueQuantity.value"] > 8][["subject.reference", "effective", "valueQuantity.value"]]

,subject.reference,effective,valueQuantity.value
0,Patient/carol,2024-11-20 00:00:00+00:00,9.1
2,urn:uuid:8f0c6a3e-0d1b-4f3a-9d61-1a2b3c4d5e01,2025-02-10 23:00:00+00:00,8.4


In [22]:
# Any of several codes, e.g. HbA1c or body weight
observations[has_code(observations["code.coding"], {"4548-4", "29463-7"}, LOINC)][["code.text", "valueQuantity.value"]]

,code.text,valueQuantity.value
0,Hemoglobin A1c/Hemoglobin.total in Blood,9.1
1,Hemoglobin A1c/Hemoglobin.total in Blood,7.8
2,Hemoglobin A1c/Hemoglobin.total in Blood,8.4
3,Hemoglobin A1c/Hemoglobin.total in Blood,5.6
4,Body weight,82.0


In [23]:
# Final laboratory results within a date window
observations[
    (observations["status"] == "final")
    & has_category(observations["category"], "laboratory")
    & observations["effective"].between("2024-01-01", "2025-01-01", inclusive="left")
][["code.text", "effective", "valueQuantity.value"]]

,code.text,effective,valueQuantity.value
0,Hemoglobin A1c/Hemoglobin.total in Blood,2024-11-20 00:00:00+00:00,9.1
1,Hemoglobin A1c/Hemoglobin.total in Blood,2024-03-04 22:30:00+00:00,7.8


In [24]:
# Every code used (shows what you can filter on)
codes(observations["code.coding"]).explode().value_counts()

code.coding
4548-4     4
29463-7    1
Name: count, dtype: int64

## 5. Joining resources to their patient

`with_patient` follows `subject.reference` to the Patient and adds `patient_*` columns. It handles both `Patient/123` and `urn:uuid:...` references.

In [25]:
hba1c_with_patient = with_patient(hba1c, patients)
hba1c_with_patient[["patient_family", "patient_given", "patient_age", "effective", "valueQuantity.value"]]

,patient_family,patient_given,patient_age,effective,valueQuantity.value
0,Brown,Carol,77,2024-11-20 00:00:00+00:00,9.1
1,Nguyen,Alice May,71,2024-03-04 22:30:00+00:00,7.8
2,Nguyen,Alice May,71,2025-02-10 23:00:00+00:00,8.4
3,Smith,Bob,46,2025-08-31 22:15:00+00:00,5.6


In [26]:
# Latest HbA1c per patient
latest_per_patient(hba1c_with_patient, "effective")[["patient_id", "patient_family", "effective", "valueQuantity.value"]]

,patient_id,patient_family,effective,valueQuantity.value
0,carol,Brown,2024-11-20 00:00:00+00:00,9.1
1,alice,Nguyen,2025-02-10 23:00:00+00:00,8.4
2,bob,Smith,2025-08-31 22:15:00+00:00,5.6


In [27]:
# Number of observations per patient
with_patient(observations, patients).groupby(["patient_id", "patient_family"]).size().sort_values(ascending=False)

patient_id  patient_family
alice       Nguyen            2
bob         Smith             2
carol       Brown             1
dtype: int64

## 6. Conditions and simple cohorts

In [28]:
conditions["code.text"].value_counts()

code.text
Diabetes mellitus type 2        2
Malignant neoplastic disease    1
Name: count, dtype: int64

In [29]:
# Active type 2 diabetes (SNOMED 44054006)
active = has_code(conditions["clinicalStatus.coding"], "active")
t2dm = with_patient(conditions[active & has_code(conditions["code.coding"], "44054006", SNOMED)], patients)
t2dm[["patient_family", "patient_age", "code.text", "onsetDateTime"]]

,patient_family,patient_age,code.text,onsetDateTime
0,Nguyen,71,Diabetes mellitus type 2,2015-04-01


In [30]:
# Cohort: active T2DM AND any HbA1c > 8
high_a1c_ids = set(hba1c_with_patient.loc[hba1c_with_patient["valueQuantity.value"] > 8, "patient_id"])
cohort = patients[patients["id"].isin(set(t2dm["patient_id"]) & high_a1c_ids)]
cohort[["id", "family", "given", "age"]]

,id,family,given,age
1,alice,Nguyen,Alice May,71


## 7. Medications

In [31]:
active_meds = with_patient(medreqs[medreqs["status"] == "active"], patients)
active_meds[["patient_family", "medicationCodeableConcept.text", "authoredOn"]]

,patient_family,medicationCodeableConcept.text,authoredOn
0,Nguyen,Metformin 500 mg tablet,2015-04-02


In [32]:
# Free-text match, handy when medication coding is inconsistent
active_meds[active_meds["medicationCodeableConcept.text"].str.contains("metformin", case=False, na=False)]

,_source,_fullUrl,resourceType,id,status,intent,authoredOn,medicationCodeableConcept.text,subject.reference,patient_id,patient_family,patient_given,patient_gender,patient_age,patient_birthDate
0,../data/sample/bundle-transaction.json#0,urn:uuid:m1,MedicationRequest,m1,active,order,2015-04-02,Metformin 500 mg tablet,urn:uuid:8f0c6a3e-0d1b-4f3a-9d61-1a2b3c4d5e01,alice,Nguyen,Alice May,female,71,1955-06-14


## Going further

- **Raw JSON** for any Bundle: `result.bundles[key]`, a plain dict.
- **Export** a result for other tools: `cohort.to_csv("cohort.csv")`. Write anything containing real patient data under `data/` (it's git-ignored), never into the repo.
- **Before committing this notebook**, make sure outputs are stripped. The README explains how to set up `nbstripout` so it happens automatically.